In [ ]:
# Keeping Team 6 weather and subway data seperate from the ride-demand results
from pathlib import Path
from functools import reduce
from datetime import datetime, timezone
import hashlib
import json
import zipfile
import pandas as pd
from pyspark.sql import functions as F, Window

# Reusing saved tables by default, enabling a build only when its outputs are missing
BUILD_BASE_WEATHER = False
BUILD_SUBWAY = False
BUILD_WEATHER_EXTENSION = False
EXPORT_HANDOFF = False

spark.conf.set("spark.sql.session.timeZone", "UTC")
NAMESPACE = "workspace.citibike_weather"
VOLUME_ROOT = "/Volumes/workspace/citibike_weather/transfer"
STATION_SOURCE_TABLE = f"{NAMESPACE}.station_information_snapshot"
CLASS_TABLE = f"{NAMESPACE}.archive_v1_weather_classes"
MAPPING_TABLE = f"{NAMESPACE}.archive_v1_station_weather_mapping"
FORECAST_VIEW = f"{NAMESPACE}.archive_v1_station_forecast"
V2_CLASS_TABLE = f"{NAMESPACE}.archive_v2_weather_classes"
V2_MAPPING_TABLE = f"{NAMESPACE}.archive_v2_station_weather_mapping"
V2_FORECAST_VIEW = f"{NAMESPACE}.archive_v2_station_forecast"
BRIDGE_TABLE = "workspace.citibike_subway.subway_complex_bike_bridge_200m_v1"
RETROSPECTIVE_TABLE = (
    "workspace.citibike_subway.archive_v1_station_hour_disruption_retrospective"
)
ALERT_FEATURE_TABLE = (
    "workspace.citibike_subway.archive_v1_station_origin_alert_features"
)

# Keeping the full saved archives and recording our agreed modelling period
STUDY_START_UTC = "2025-03-01 05:00:00"
STUDY_END_UTC_EXCLUSIVE = "2026-08-11 04:00:00"
HORIZONS = [1, 2, 3, 4, 5, 6]
MODELING_HORIZONS = [1, 2, 3]
EXPECTED_CLASS_COUNT = 28
ASSUMED_AVAILABILITY_DELAY_HOURS = 7
MAX_MAPPING_DISTANCE_KM = 10.0
ALLOW_RELOCATED_STATIONS = False
STATION_AS_OF_UTC = None
ORIGIN_START_UTC = None
ORIGIN_END_UTC = None
PACKAGE_PATH = Path(VOLUME_ROOT) / "citibike_weather_input.zip"

# Keeping the original assumptions beside the code, without using future exposure
# Treating early ECMWF hindcasts as retrospective forecasts, with assumed availability
# Linking each subway location to its nearest dock within 200 m, not every nearby dock
# Keeping station geography retrospective; historical openings and moves are limitations
# Using recent route publications, not confirmed service loss at each dock
# Keeping three-hour publication windows and zero assumed ingestion delay
# Keeping archive-silence event counts null, and leaving unmapped docks unmapped
# Keeping the retrospective exposure table out of forecasting predictors
# Preserving earlier cell outputs as records of completed runs, not new executions


In [ ]:
# Checking build destinations before starting any optional processing
build_groups = [
    (BUILD_BASE_WEATHER, [CLASS_TABLE, MAPPING_TABLE, FORECAST_VIEW]),
    (BUILD_SUBWAY, [RETROSPECTIVE_TABLE, ALERT_FEATURE_TABLE]),
    (BUILD_WEATHER_EXTENSION, [V2_CLASS_TABLE, V2_MAPPING_TABLE, V2_FORECAST_VIEW]),
]
for enabled, destinations in build_groups:
    if enabled:
        existing_destinations = [
            name for name in destinations if spark.catalog.tableExists(name)
        ]
        assert not existing_destinations, (
            f"Keeping existing outputs: {existing_destinations}. "
            "Leave this build switch False to reuse them."
        )

if BUILD_BASE_WEATHER:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {NAMESPACE}")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {NAMESPACE}.transfer")

if BUILD_SUBWAY:
    assert spark.catalog.tableExists(BRIDGE_TABLE), (
        "The validated 200 m bridge is an input; create or import it first."
    )

def assert_empty(frame, message):
    assert not frame.limit(1).count(), message

def assert_unique(frame, keys, label):
    assert_empty(
        frame.groupBy(*keys).count().filter(F.col("count") != 1),
        f"Duplicate {label}: {keys}",
    )

# Keeping completed extraction and mapping skipped on a normal rerun
print("Base weather build:", BUILD_BASE_WEATHER)
print("Subway build:", BUILD_SUBWAY)
print("Weather extension build:", BUILD_WEATHER_EXTENSION)


In [0]:
if BUILD_BASE_WEATHER and not spark.catalog.tableExists(STATION_SOURCE_TABLE):
    # Reading the downloaded station information snapshot.
    station_file = (
        "/Volumes/workspace/citibike_weather/transfer/"
        "station_information.json"
    )

    station_feed = spark.read.option("multiLine", "true").json(station_file)

    # Expanding the station array into one row per station.
    station_records = (
        station_feed
        .select(F.explode("data.stations").alias("station"))
        .select("station.*")
    )

    print("Station records:", station_records.count())

    display(
        station_records.select(
            "station_id", "name", "lat", "lon", "capacity"
        ).limit(10)
    )


Station records: 2519


station_id,name,lat,lon,capacity
0687075b-cf2d-4130-a859-fe841aafbd3f,Vernon Blvd & 41 Rd,40.7552433,-73.9485678,27
2206779090602992084,68 Dr & 112 St,40.72704,-73.84128,12
2250952758815752682,Dumont Ave & Barbey St,40.66759,-73.88609,12
c2d549f8-f54c-4a0d-9ecd-6e6f79118279,Cooper Ave & 65 Pl,40.69953,-73.88873,23
66dd26f9-0aca-11e7-82f6-3863bb44ef7c,9 St & 44 Rd,40.74966,-73.9521,34
595cf79f-7783-4373-b3d5-775bcf76033b,Ogden Ave & Merriam Ave,40.839142,-73.925961,19
6fb81fda-f0b5-4242-90f9-88d0214feb8d,Sedgwick Ave & W Tremont Ave,40.85304455868985,-73.91830623149872,19
1917312704757770810,1 Ave & E 38 St,40.746202,-73.971822,39
ef6216aa-d3c8-46bb-8c6b-b62f8ba51b16,Van Sinderen Ave & Truxton St,40.6786,-73.90369,23
d979131e-0b9f-4aa2-88c5-46d4ff94abc8,Pacific St & Sackman St,40.67522,-73.9061,19


In [0]:
if BUILD_BASE_WEATHER and not spark.catalog.tableExists(STATION_SOURCE_TABLE):
    # Standardizing the fields needed for weather mapping.
    stations_clean = station_records.select(
        F.trim(F.col("station_id").cast("string")).alias("station_id"),
        F.col("name").cast("string").alias("name"),
        F.col("lat").cast("double").alias("lat"),
        F.col("lon").cast("double").alias("lon"),
    )

    # Checking identifiers and geographic coordinates.
    station_quality = stations_clean.agg(
        F.count("*").alias("rows"),
        F.countDistinct("station_id").alias("unique_station_ids"),
        F.sum(
            F.when(
                F.col("station_id").isNull() | (F.col("station_id") == ""),
                1,
            ).otherwise(0)
        ).alias("missing_ids"),
        F.sum(
            F.when(
                F.col("lat").isNull()
                | F.col("lon").isNull()
                | F.isnan("lat")
                | F.isnan("lon")
                | ~F.col("lat").between(-90, 90)
                | ~F.col("lon").between(-180, 180),
                1,
            ).otherwise(0)
        ).alias("invalid_coordinates"),
    )

    display(station_quality)

    quality = station_quality.first()

    assert quality["missing_ids"] == 0, "Missing station IDs."
    assert quality["unique_station_ids"] == quality["rows"], "Duplicate station IDs."
    assert quality["invalid_coordinates"] == 0, "Invalid station coordinates."


rows,unique_station_ids,missing_ids,invalid_coordinates
2519,2519,0,0


In [0]:
if BUILD_BASE_WEATHER and not spark.catalog.tableExists(STATION_SOURCE_TABLE):
    station_table = STATION_SOURCE_TABLE

    # Recording the snapshot's import time and source.
    station_snapshot = (
        stations_clean
        .withColumn("fetched_at", F.current_timestamp())
        .withColumn("source_file", F.lit(station_file))
        .withColumn("snapshot_timestamp_basis", F.lit("import_time"))
    )

    # Making the validated records available to SQL.
    station_snapshot.createOrReplaceTempView("station_snapshot_to_save")

    # Creating the Delta table without replacing an existing table.
    spark.sql(f"""
        CREATE TABLE {station_table}
        USING DELTA
        AS SELECT * FROM station_snapshot_to_save
    """)

    print("Saved table:", station_table)
    print("Saved stations:", spark.table(station_table).count())


Saved table: workspace.citibike_weather.station_information_snapshot
Saved stations: 2519


In [0]:
if BUILD_BASE_WEATHER:
    # Verifying the package before placing its contents in the managed volume.
    assert PACKAGE_PATH.is_file(), f"Upload the local package first: {PACKAGE_PATH}"
    package_sha256 = hashlib.file_digest(PACKAGE_PATH.open("rb"), "sha256").hexdigest()
    extract_root = Path(VOLUME_ROOT) / "inputs" / package_sha256
    with zipfile.ZipFile(PACKAGE_PATH) as archive:
        assert len(archive.namelist()) == len(set(archive.namelist())), "Duplicate ZIP members."
        manifest = json.loads(archive.read("input_manifest.json"))
        assert manifest["format_version"] == 1
        records = manifest["files"]
        assert records and len(records) == len({r["path"] for r in records})
        assert set(archive.namelist()) == {r["path"] for r in records} | {"input_manifest.json"}
        for record in records:
            relative = Path(record["path"])
            assert not relative.is_absolute() and ".." not in relative.parts
            assert relative.parts[0] == "weather_classes" and relative.suffix == ".parquet"
            payload = archive.read(record["path"])
            assert len(payload) == record["size"]
            assert hashlib.sha256(payload).hexdigest() == record["sha256"]
            destination = extract_root / relative
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(payload)
    WEATHER_INPUT_ROOT = str(extract_root / "weather_classes")
    print("Verified monthly files:", len(records))
    print("Package SHA-256:", package_sha256)
    print("Spark input:", WEATHER_INPUT_ROOT)


Verified monthly files: 23
Package SHA-256: bf65a4cc1d0135537b005d61c076af3695355447c3c13ebd0bec6955b7b077a9
Spark input: /Volumes/workspace/citibike_weather/transfer/inputs/bf65a4cc1d0135537b005d61c076af3695355447c3c13ebd0bec6955b7b077a9/weather_classes


In [0]:
if BUILD_BASE_WEATHER:
    # Reading only the completed Parquet files and preserving their source paths.
    weather = (spark.read.option("recursiveFileLookup", "true")
               .option("pathGlobFilter", "selected_weather_request_classes_*.parquet")
               .parquet(WEATHER_INPUT_ROOT))
    WEATHER_VALUES = ["forecast_temperature_f", "forecast_precipitation_in",
                      "forecast_rain_in", "forecast_snowfall_in", "forecast_wind_speed_mph"]
    TIMESTAMPS = ["forecast_origin_utc", "target_hour_utc", "selected_model_run_utc",
                  "selected_run_available_utc", "valid_time_utc"]
    SCHEDULE_COLUMNS = ["target_hour_utc", "selected_model_run_utc", "selected_run_available_utc",
                        "selected_run_age_hours", "weather_lead_hours", "is_fallback_weather_run",
                        "selected_forecast_hours", "selected_weather_source", "selected_raw_json_relative_path"]
    REQUIRED = (TIMESTAMPS + WEATHER_VALUES + ["weather_request_class", "forecast_horizon",
                "representative_location_id", "request_latitude", "request_longitude",
                "model_grid_latitude", "model_grid_longitude", "selected_run_age_hours",
                "weather_lead_hours", "is_fallback_weather_run", "selected_forecast_hours",
                "selected_weather_source", "selected_raw_json_relative_path"])
    assert not (set(REQUIRED) - set(weather.columns)), sorted(set(REQUIRED) - set(weather.columns))
    weather = weather.select(*REQUIRED)
    for name in TIMESTAMPS:
        weather = weather.withColumn(name, F.col(name).cast("timestamp"))
    for name in WEATHER_VALUES:
        weather = weather.withColumn(name, F.when(F.isnan(F.col(name)), None).otherwise(F.col(name)).cast("float"))
    for name in ["weather_request_class", "forecast_horizon", "representative_location_id",
                 "selected_run_age_hours", "weather_lead_hours", "selected_forecast_hours"]:
        weather = weather.withColumn(name, F.col(name).cast("int"))
    weather = weather.withColumn("is_fallback_weather_run", F.col("is_fallback_weather_run").cast("boolean"))
    if ORIGIN_START_UTC is not None:
        weather = weather.filter(F.col("forecast_origin_utc") >= F.to_timestamp(F.lit(ORIGIN_START_UTC)))
    if ORIGIN_END_UTC is not None:
        weather = weather.filter(F.col("forecast_origin_utc") < F.to_timestamp(F.lit(ORIGIN_END_UTC)))
    weather = (weather.withColumn("origin_year", F.year("forecast_origin_utc"))
               .withColumn("origin_month", F.month("forecast_origin_utc"))
               .withColumn("weather_forecast_has_missing", reduce(lambda a, b: a | b,
                            [F.col(name).isNull() for name in WEATHER_VALUES])))


root
 |-- forecast_origin_utc: timestamp (nullable = true)
 |-- target_hour_utc: timestamp (nullable = true)
 |-- selected_model_run_utc: timestamp (nullable = true)
 |-- selected_run_available_utc: timestamp (nullable = true)
 |-- valid_time_utc: timestamp (nullable = true)
 |-- forecast_temperature_f: float (nullable = true)
 |-- forecast_precipitation_in: float (nullable = true)
 |-- forecast_rain_in: float (nullable = true)
 |-- forecast_snowfall_in: float (nullable = true)
 |-- forecast_wind_speed_mph: float (nullable = true)
 |-- weather_request_class: integer (nullable = true)
 |-- forecast_horizon: integer (nullable = true)
 |-- representative_location_id: integer (nullable = true)
 |-- request_latitude: double (nullable = true)
 |-- request_longitude: double (nullable = true)
 |-- model_grid_latitude: double (nullable = true)
 |-- model_grid_longitude: double (nullable = true)
 |-- selected_run_age_hours: integer (nullable = true)
 |-- weather_lead_hours: integer (nullable = t

forecast_origin_utc,target_hour_utc,selected_model_run_utc,selected_run_available_utc,valid_time_utc,forecast_temperature_f,forecast_precipitation_in,forecast_rain_in,forecast_snowfall_in,forecast_wind_speed_mph,weather_request_class,forecast_horizon,representative_location_id,request_latitude,request_longitude,model_grid_latitude,model_grid_longitude,selected_run_age_hours,weather_lead_hours,is_fallback_weather_run,selected_forecast_hours,selected_weather_source,selected_raw_json_relative_path,origin_year,origin_month,weather_forecast_has_missing
2025-05-01T00:00:00.000Z,2025-05-01T01:00:00.000Z,2025-04-30T12:00:00.000Z,2025-04-30T19:00:00.000Z,2025-05-01T01:00:00.000Z,65.6,0.0,0.0,0.0,7.3,0,1,44,40.52549110299151,-74.23353546529498,40.52724,-74.237274,12,13,false,19,normal_archived_run,data\bronze\weather_forecasts\open_meteo_single_runs\ecmwf_ifs\request_classes=28\forecast_hours=19\year=2025\month=04\ecmwf_ifs_20250430T1200Z.json,2025,5,false
2025-05-01T00:00:00.000Z,2025-05-01T01:00:00.000Z,2025-04-30T12:00:00.000Z,2025-04-30T19:00:00.000Z,2025-05-01T01:00:00.000Z,64.8,0.0,0.0,0.0,6.9,1,1,109,40.55186202658534,-74.15089028772675,40.597538,-74.215,12,13,false,19,normal_archived_run,data\bronze\weather_forecasts\open_meteo_single_runs\ecmwf_ifs\request_classes=28\forecast_hours=19\year=2025\month=04\ecmwf_ifs_20250430T1200Z.json,2025,5,false
2025-05-01T00:00:00.000Z,2025-05-01T01:00:00.000Z,2025-04-30T12:00:00.000Z,2025-04-30T19:00:00.000Z,2025-05-01T01:00:00.000Z,64.9,0.0,0.0,0.0,6.9,2,1,5,40.55265878171833,-74.18848459684874,40.597538,-74.215,12,13,false,19,normal_archived_run,data\bronze\weather_forecasts\open_meteo_single_runs\ecmwf_ifs\request_classes=28\forecast_hours=19\year=2025\month=04\ecmwf_ifs_20250430T1200Z.json,2025,5,false
2025-05-01T00:00:00.000Z,2025-05-01T01:00:00.000Z,2025-04-30T12:00:00.000Z,2025-04-30T19:00:00.000Z,2025-05-01T01:00:00.000Z,65.4,0.0,0.0,0.0,7.7,3,1,176,40.56199406192752,-74.12258304615597,40.597538,-74.08771,12,13,false,19,normal_archived_run,data\bronze\weather_forecasts\open_meteo_single_runs\ecmwf_ifs\request_classes=28\forecast_hours=19\year=2025\month=04\ecmwf_ifs_20250430T1200Z.json,2025,5,false
2025-05-01T00:00:00.000Z,2025-05-01T01:00:00.000Z,2025-04-30T12:00:00.000Z,2025-04-30T19:00:00.000Z,2025-05-01T01:00:00.000Z,65.1,0.0,0.0,0.0,7.7,4,1,14,40.62483367348954,-74.0298925091807,40.597538,-74.08771,12,13,false,19,normal_archived_run,data\bronze\weather_forecasts\open_meteo_single_runs\ecmwf_ifs\request_classes=28\forecast_hours=19\year=2025\month=04\ecmwf_ifs_20250430T1200Z.json,2025,5,false
2025-05-01T00:00:00.000Z,2025-05-01T01:00:00.000Z,2025-04-30T12:00:00.000Z,2025-04-30T19:00:00.000Z,2025-05-01T01:00:00.000Z,65.4,0.0,0.0,0.0,7.7,5,1,110,40.54577972951617,-74.1283431099553,40.597538,-74.08771,12,13,false,19,normal_archived_run,data\bronze\weather_forecasts\open_meteo_single_runs\ecmwf_ifs\request_classes=28\forecast_hours=19\year=2025\month=04\ecmwf_ifs_20250430T1200Z.json,2025,5,false


In [0]:
if BUILD_BASE_WEATHER:
    # Checking required metadata, timestamp arithmetic, and completeness of the schedule.
    nonnull = [name for name in REQUIRED if name not in WEATHER_VALUES]
    assert_empty(weather.filter(reduce(lambda a, b: a | b,
                 [F.col(name).isNull() for name in nonnull])), "Missing required weather metadata.")
    assert_unique(weather, ["forecast_origin_utc", "forecast_horizon", "weather_request_class"], "weather keys")
    origin_seconds = F.col("forecast_origin_utc").cast("long")
    target_seconds = F.col("target_hour_utc").cast("long")
    run_seconds = F.col("selected_model_run_utc").cast("long")
    available_seconds = F.col("selected_run_available_utc").cast("long")
    bad_time = (
        (target_seconds != origin_seconds + F.col("forecast_horizon") * 3600)
        | (target_seconds != run_seconds + F.col("weather_lead_hours") * 3600)
        | (origin_seconds != run_seconds + F.col("selected_run_age_hours") * 3600)
        | (available_seconds != run_seconds + ASSUMED_AVAILABILITY_DELAY_HOURS * 3600)
        | (available_seconds > origin_seconds)
        | (F.col("valid_time_utc") != F.col("target_hour_utc"))
        | (F.col("weather_lead_hours") >= F.col("selected_forecast_hours"))
        | (F.col("weather_lead_hours") < 0)
        | (origin_seconds % 3600 != 0)
        | (~F.col("forecast_horizon").isin(HORIZONS))
        | (F.abs(target_seconds - F.col("target_hour_utc").cast("double")) > 0)
    )
    assert_empty(weather.filter(bad_time), "Weather timing or horizon contract failed.")
    coverage = weather.groupBy("forecast_origin_utc", "forecast_horizon").agg(
        F.countDistinct("weather_request_class").alias("classes"),
        F.countDistinct(F.struct(*SCHEDULE_COLUMNS)).alias("schedules"))
    assert_empty(coverage.filter((F.col("classes") != EXPECTED_CLASS_COUNT)
                                 | (F.col("schedules") != 1)), "Class coverage or schedule agreement failed.")
    assert_empty(coverage.groupBy("forecast_origin_utc").count().filter(F.col("count") != len(HORIZONS)),
                 "Missing horizon at an origin.")
    summary = weather.agg(F.count("*").alias("rows"),
        F.countDistinct("forecast_origin_utc").alias("origins"),
        F.min("forecast_origin_utc").alias("first_origin"),
        F.max("forecast_origin_utc").alias("last_origin"),
        F.sum(F.col("weather_forecast_has_missing").cast("long")).alias("incomplete_rows")).first()
    assert summary.rows > 0, "No forecasts in selected interval."
    assert summary.rows == summary.origins * len(HORIZONS) * EXPECTED_CLASS_COUNT
    assert int((summary.last_origin - summary.first_origin).total_seconds() / 3600) + 1 == summary.origins, "Missing hourly origins."
    assert weather.select("weather_request_class").distinct().count() == EXPECTED_CLASS_COUNT
    print(summary.asDict())
    display(weather.groupBy("origin_year", "origin_month").agg(
        F.count("*").alias("class_rows"),
        F.sum(F.col("weather_forecast_has_missing").cast("long")).alias("incomplete_rows"),
        F.sum(F.col("is_fallback_weather_run").cast("long")).alias("fallback_class_rows")
    ).orderBy("origin_year", "origin_month"))


{'rows': 2780568, 'origins': 16551, 'first_origin': datetime.datetime(2024, 8, 10, 7, 0), 'last_origin': datetime.datetime(2026, 6, 30, 21, 0), 'incomplete_rows': 23184}


origin_year,origin_month,class_rows,incomplete_rows,fallback_class_rows
2024,8,87528,0,0
2024,9,120960,0,0
2024,10,124992,0,0
2024,11,120960,0,0
2024,12,124992,0,0
2025,1,124992,0,0
2025,2,112896,0,0
2025,3,124992,0,0
2025,4,120960,0,0
2025,5,124992,0,0


In [0]:
if BUILD_BASE_WEATHER:
    # Selecting station snapshots without hiding invalid latest locations behind older valid records.
    assert spark.catalog.tableExists(STATION_SOURCE_TABLE), f"Station table not found: {STATION_SOURCE_TABLE}"
    stations_raw = spark.table(STATION_SOURCE_TABLE)
    assert {"station_id", "name", "lat", "lon", "fetched_at"} <= set(stations_raw.columns)
    stations_raw = stations_raw.select(
        F.trim(F.col("station_id").cast("string")).alias("station_id"),
        F.col("name").cast("string").alias("station_name"),
        F.col("lat").cast("double").alias("station_latitude"),
        F.col("lon").cast("double").alias("station_longitude"),
        F.col("fetched_at").cast("timestamp").alias("station_metadata_fetched_at_utc"))
    assert_empty(stations_raw.filter(F.col("station_id").isNull() | (F.col("station_id") == "")
        | F.col("station_metadata_fetched_at_utc").isNull()), "Missing station ID or collection timestamp.")
    if STATION_AS_OF_UTC is not None:
        stations_raw = stations_raw.filter(F.col("station_metadata_fetched_at_utc") <= F.to_timestamp(F.lit(STATION_AS_OF_UTC)))
    station_cutoff = stations_raw.agg(F.max("station_metadata_fetched_at_utc")).first()[0]
    assert station_cutoff is not None, "No station snapshots at this cutoff."
    latest_times = stations_raw.groupBy("station_id").agg(
        F.max("station_metadata_fetched_at_utc").alias("station_metadata_fetched_at_utc"))
    latest = stations_raw.join(latest_times, ["station_id", "station_metadata_fetched_at_utc"], "inner").distinct()
    assert_unique(latest, ["station_id"], "latest station records (conflicting same-time metadata)")
    invalid_station = (F.col("station_latitude").isNull() | F.isnan("station_latitude")
        | F.col("station_longitude").isNull() | F.isnan("station_longitude")
        | ~F.col("station_latitude").between(-90, 90)
        | ~F.col("station_longitude").between(-180, 180))
    assert_empty(latest.filter(invalid_station), "Invalid coordinates in the latest station snapshot.")
    location_versions = stations_raw.groupBy("station_id").agg(F.countDistinct(F.struct(
        F.round("station_latitude", 5), F.round("station_longitude", 5))).alias("station_location_version_count"))
    stations = latest.join(location_versions, "station_id")
    relocated_count = stations.filter(F.col("station_location_version_count") > 1).count()
    print("Selected station cutoff (UTC):", station_cutoff)
    print("Stations with changed coordinates in available snapshots:", relocated_count)
    assert ALLOW_RELOCATED_STATIONS or relocated_count == 0, "Review relocations before enabling ALLOW_RELOCATED_STATIONS."
    station_count = stations.count()
    assert station_count > 0
    print("Stations to map:", station_count)
    display(stations.orderBy("station_id").limit(10))


Selected station cutoff (UTC): 2026-10-18 02:42:08.995720
Stations with changed coordinates in available snapshots: 0
Stations to map: 2519


station_id,station_metadata_fetched_at_utc,station_name,station_latitude,station_longitude,station_location_version_count
00284700-9d22-42ce-8485-113fed9879c1,2026-10-18T02:42:08.995Z,28 Ave & 44 St,40.76408932350688,-73.91065120697021,1
002e6e9f-ced2-47b0-bc7e-ccf38c3b4b0f,2026-10-18T02:42:08.995Z,Prospect Ave & E 151 St,40.814232,-73.903927,1
00967b8f-1a4d-4131-a65a-17fa8ca89e28,2026-10-18T02:42:08.995Z,41 Ave & 67 St,40.74446,-73.89764,1
0096fecb-3a56-4585-9204-aa9554f7bc49,2026-10-18T02:42:08.995Z,E 5 St & Ave A,40.72479007746337,-73.98430064320564,1
00d3e65e-b614-4408-aaa9-939ed96ce645,2026-10-18T02:42:08.995Z,Perry St & Greenwich Ave,40.735918,-74.000939,1
00d94dfc-3695-47e1-aba6-7f319dd2eb08,2026-10-18T02:42:08.995Z,Bergen St & Kingston Ave,40.6756,-73.9415,1
0109f17a-2378-4500-9648-88bdb31ec612,2026-10-18T02:42:08.995Z,Degraw St & 3 Ave,40.67977,-73.9847,1
0138452a-b9f4-4aee-80b0-7fae9a122ffe,2026-10-18T02:42:08.995Z,E 54 St & 1 Ave,40.756265,-73.964179,1
014a931b-8e1c-438a-8a22-f267ddd1e0d2,2026-10-18T02:42:08.995Z,Peck Slip & South St,40.7075188830923,-74.00108054280281,1
016ee47a-4871-4209-91cb-bf3ceeddc991,2026-10-18T02:42:08.995Z,St Felix Ave & 61 St,40.69715,-73.8936,1


In [0]:
if BUILD_BASE_WEATHER:
    # Building a small, stable location lookup before joining it to the time series.
    # Standardizing coordinate precision for the mapping lookup only.
    representatives = (
        weather.select(
            "weather_request_class",
            "representative_location_id",
            F.round("request_latitude", 7).alias("request_latitude"),
            F.round("request_longitude", 7).alias("request_longitude"),
        )
        .distinct()
    )
    assert_unique(representatives, ["weather_request_class"], "representative coordinates")
    assert_empty(representatives.filter(F.col("request_latitude").isNull()
        | F.isnan("request_latitude") | ~F.col("request_latitude").between(-90, 90)
        | F.col("request_longitude").isNull() | F.isnan("request_longitude")
        | ~F.col("request_longitude").between(-180, 180)), "Invalid representative coordinates.")
    pairs = stations.crossJoin(F.broadcast(representatives))
    a = (F.pow(F.sin(F.radians(F.col("request_latitude") - F.col("station_latitude")) / 2), 2)
         + F.cos(F.radians("station_latitude")) * F.cos(F.radians("request_latitude"))
         * F.pow(F.sin(F.radians(F.col("request_longitude") - F.col("station_longitude")) / 2), 2))
    pairs = pairs.withColumn("weather_mapping_distance_km",
        6371.0088 * 2 * F.asin(F.sqrt(F.least(F.lit(1.0), F.greatest(F.lit(0.0), a)))))
    rank = Window.partitionBy("station_id").orderBy("weather_mapping_distance_km", "weather_request_class")
    mapping = (pairs.withColumn("_rank", F.row_number().over(rank)).filter(F.col("_rank") == 1).drop("_rank")
        .withColumn("weather_mapping_method", F.lit("nearest_archived_request_coordinate"))
        .withColumn("station_geography_policy", F.lit("static_retrospective_backfill"))
        .withColumn("station_snapshot_cutoff_utc", F.lit(station_cutoff))
        .withColumn("input_package_sha256", F.lit(package_sha256)))
    assert_unique(mapping, ["station_id"], "station mappings")
    assert mapping.count() == station_count
    print("Most distant station assignments:")
    display(mapping.orderBy(F.desc("weather_mapping_distance_km")).limit(15))
    assert_empty(mapping.filter(F.col("weather_mapping_distance_km") > MAX_MAPPING_DISTANCE_KM),
                 "Stations exceed the configured mapping distance. Review geographic coverage.")
    print("All stations mapped within the configured distance.")
    print("Expected station forecast rows:", summary.origins * len(HORIZONS) * station_count)


Most distant station assignments:


station_id,station_metadata_fetched_at_utc,station_name,station_latitude,station_longitude,station_location_version_count,weather_request_class,representative_location_id,request_latitude,request_longitude,weather_mapping_distance_km,weather_mapping_method,station_geography_policy,station_snapshot_cutoff_utc,input_package_sha256
1843486301383097150,2026-10-18T02:42:08.995Z,41 Ave & 80 St,40.74586,-73.88539,1,21,4,40.7237521,-73.9769683,8.098223033347221,nearest_archived_request_coordinate,static_retrospective_backfill,2026-10-18T02:42:08.995Z,bf65a4cc1d0135537b005d61c076af3695355447c3c13ebd0bec6955b7b077a9
1836581441921155232,2026-10-18T02:42:08.995Z,79 St & 37 Ave,40.74937,-73.88714,1,21,4,40.7237521,-73.9769683,8.086761782418918,nearest_archived_request_coordinate,static_retrospective_backfill,2026-10-18T02:42:08.995Z,bf65a4cc1d0135537b005d61c076af3695355447c3c13ebd0bec6955b7b077a9
1843486390605672640,2026-10-18T02:42:08.995Z,Ithaca St & Baxter Ave,40.74638,-73.88322,1,22,9,40.7510343,-73.7879488,8.042232807738845,nearest_archived_request_coordinate,static_retrospective_backfill,2026-10-18T02:42:08.995Z,bf65a4cc1d0135537b005d61c076af3695355447c3c13ebd0bec6955b7b077a9
1838617691511229014,2026-10-18T02:42:08.995Z,81 St & 35 Ave,40.75179,-73.88568,1,25,41,40.8043339,-73.951292,8.041024778616311,nearest_archived_request_coordinate,static_retrospective_backfill,2026-10-18T02:42:08.995Z,bf65a4cc1d0135537b005d61c076af3695355447c3c13ebd0bec6955b7b077a9
1843486189591985058,2026-10-18T02:42:08.995Z,79 St & Roosevelt Ave,40.74718,-73.88675,1,21,4,40.7237521,-73.9769683,8.0354232583718,nearest_archived_request_coordinate,static_retrospective_backfill,2026-10-18T02:42:08.995Z,bf65a4cc1d0135537b005d61c076af3695355447c3c13ebd0bec6955b7b077a9
1906248765431831024,2026-10-18T02:42:08.995Z,82 St & Broadway,40.74281,-73.88296,1,11,17,40.691507,-73.9499048,8.02344159595579,nearest_archived_request_coordinate,static_retrospective_backfill,2026-10-18T02:42:08.995Z,bf65a4cc1d0135537b005d61c076af3695355447c3c13ebd0bec6955b7b077a9
2009671276352543716,2026-10-18T02:42:08.995Z,34 Ave & 83 St,40.753935004830225,-73.88390148299196,1,25,41,40.8043339,-73.951292,7.975208217162462,nearest_archived_request_coordinate,static_retrospective_backfill,2026-10-18T02:42:08.995Z,bf65a4cc1d0135537b005d61c076af3695355447c3c13ebd0bec6955b7b077a9
1828967830172187422,2026-10-18T02:42:08.995Z,84 St & Roosevelt Ave,40.74797,-73.88204,1,22,9,40.7510343,-73.7879488,7.933386926146123,nearest_archived_request_coordinate,static_retrospective_backfill,2026-10-18T02:42:08.995Z,bf65a4cc1d0135537b005d61c076af3695355447c3c13ebd0bec6955b7b077a9
1869754328201607270,2026-10-18T02:42:08.995Z,79 St & Woodside Ave,40.74364,-73.88622,1,11,17,40.691507,-73.9499048,7.900123010316051,nearest_archived_request_coordinate,static_retrospective_backfill,2026-10-18T02:42:08.995Z,bf65a4cc1d0135537b005d61c076af3695355447c3c13ebd0bec6955b7b077a9
1828968327357059242,2026-10-18T02:42:08.995Z,85 St & Northern Blvd,40.75601,-73.88266,1,25,41,40.8043339,-73.951292,7.890964243892821,nearest_archived_request_coordinate,static_retrospective_backfill,2026-10-18T02:42:08.995Z,bf65a4cc1d0135537b005d61c076af3695355447c3c13ebd0bec6955b7b077a9


All stations mapped within the configured distance.
Expected station forecast rows: 250151814


In [0]:
if BUILD_BASE_WEATHER:
    # Persisting the reusable components and a reproducible dataset identifier.
    build_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    weather_saved = (weather.withColumn("input_package_sha256", F.lit(package_sha256))
        .withColumn("weather_model", F.lit("ecmwf_ifs"))
        .withColumn("assumed_availability_delay_hours", F.lit(ASSUMED_AVAILABILITY_DELAY_HOURS))
        .withColumn("dataset_build_id", F.lit(build_id)))
    mapping_saved = mapping.withColumn("dataset_build_id", F.lit(build_id))
    # Creating new tables without replacing earlier builds.
    weather_saved.createOrReplaceTempView("base_weather_to_save")
    mapping_saved.createOrReplaceTempView("base_weather_mapping_to_save")
    spark.sql(f"""CREATE TABLE {CLASS_TABLE} USING DELTA
    PARTITIONED BY (origin_year, origin_month)
    AS SELECT * FROM base_weather_to_save""")
    spark.sql(f"""CREATE TABLE {MAPPING_TABLE} USING DELTA
    AS SELECT * FROM base_weather_mapping_to_save""")

    # Selecting weather columns once while avoiding duplicate mapping/provenance names.
    mapping_columns = ["station_id", "station_name", "station_latitude", "station_longitude",
        "station_metadata_fetched_at_utc", "station_location_version_count", "weather_mapping_distance_km",
        "weather_mapping_method", "station_geography_policy", "station_snapshot_cutoff_utc"]
    map_sql = ", ".join(f"m.`{name}`" for name in mapping_columns)
    spark.sql(f"""CREATE VIEW {FORECAST_VIEW} AS
    SELECT {map_sql}, w.*
    FROM {MAPPING_TABLE} m
    JOIN {CLASS_TABLE} w
     ON m.weather_request_class = w.weather_request_class
     AND m.dataset_build_id = w.dataset_build_id
    """)
    print("Saved weather:", CLASS_TABLE)
    print("Saved mapping:", MAPPING_TABLE)
    print("Station forecast view:", FORECAST_VIEW)
    print("Build ID:", build_id)
    display(spark.table(FORECAST_VIEW).limit(6))


Saved weather: workspace.citibike_weather.archive_v1_weather_classes
Saved mapping: workspace.citibike_weather.archive_v1_station_weather_mapping
Station forecast view: workspace.citibike_weather.archive_v1_station_forecast
Build ID: 20261018T033852681910Z


station_id,station_name,station_latitude,station_longitude,station_metadata_fetched_at_utc,station_location_version_count,weather_mapping_distance_km,weather_mapping_method,station_geography_policy,station_snapshot_cutoff_utc,forecast_origin_utc,target_hour_utc,selected_model_run_utc,selected_run_available_utc,valid_time_utc,forecast_temperature_f,forecast_precipitation_in,forecast_rain_in,forecast_snowfall_in,forecast_wind_speed_mph,weather_request_class,forecast_horizon,representative_location_id,request_latitude,request_longitude,model_grid_latitude,model_grid_longitude,selected_run_age_hours,weather_lead_hours,is_fallback_weather_run,selected_forecast_hours,selected_weather_source,selected_raw_json_relative_path,origin_year,origin_month,weather_forecast_has_missing,input_package_sha256,weather_model,assumed_availability_delay_hours,dataset_build_id
f06c01c3-fd7f-443c-a0c7-fc9233d77f6a,67 St & Erik Pl,40.633385,-74.016562,2026-10-18T02:42:08.995Z,1,1.472986843016889,nearest_archived_request_coordinate,static_retrospective_backfill,2026-10-18T02:42:08.995Z,2025-05-01T00:00:00.000Z,2025-05-01T01:00:00.000Z,2025-04-30T12:00:00.000Z,2025-04-30T19:00:00.000Z,2025-05-01T01:00:00.000Z,65.1,0.0,0.0,0.0,7.7,4,1,14,40.62483367348954,-74.0298925091807,40.597538,-74.08771,12,13,false,19,normal_archived_run,data\bronze\weather_forecasts\open_meteo_single_runs\ecmwf_ifs\request_classes=28\forecast_hours=19\year=2025\month=04\ecmwf_ifs_20250430T1200Z.json,2025,5,false,bf65a4cc1d0135537b005d61c076af3695355447c3c13ebd0bec6955b7b077a9,ecmwf_ifs,7,20261018T033852681910Z
fd0e86e3-079d-40aa-8231-ec2cbdbd9e7f,34 St & 4 Ave,40.655278,-74.003101,2026-10-18T02:42:08.995Z,1,0.7621916192983097,nearest_archived_request_coordinate,static_retrospective_backfill,2026-10-18T02:42:08.995Z,2025-05-01T00:00:00.000Z,2025-05-01T01:00:00.000Z,2025-04-30T12:00:00.000Z,2025-04-30T19:00:00.000Z,2025-05-01T01:00:00.000Z,65.8,0.0,0.0,0.0,8.2,9,1,228,40.652354155320815,-74.0112729025337,40.597538,-73.96039,12,13,false,19,normal_archived_run,data\bronze\weather_forecasts\open_meteo_single_runs\ecmwf_ifs\request_classes=28\forecast_hours=19\year=2025\month=04\ecmwf_ifs_20250430T1200Z.json,2025,5,false,bf65a4cc1d0135537b005d61c076af3695355447c3c13ebd0bec6955b7b077a9,ecmwf_ifs,7,20261018T033852681910Z
ffaa8f6a-a4ea-4d3a-a57c-f42f77cd4581,Thomas S. Boyland St & Fulton St,40.67854,-73.91367,2026-10-18T02:42:08.995Z,1,3.3784457643042303,nearest_archived_request_coordinate,static_retrospective_backfill,2026-10-18T02:42:08.995Z,2025-05-01T00:00:00.000Z,2025-05-01T01:00:00.000Z,2025-04-30T12:00:00.000Z,2025-04-30T19:00:00.000Z,2025-05-01T01:00:00.000Z,66.6,0.0,0.0,0.0,6.7,11,1,17,40.69150702163218,-73.94990480442759,40.667835,-73.93768,12,13,false,19,normal_archived_run,data\bronze\weather_forecasts\open_meteo_single_runs\ecmwf_ifs\request_classes=28\forecast_hours=19\year=2025\month=04\ecmwf_ifs_20250430T1200Z.json,2025,5,false,bf65a4cc1d0135537b005d61c076af3695355447c3c13ebd0bec6955b7b077a9,ecmwf_ifs,7,20261018T033852681910Z
e25f8dee-c110-489f-a978-464ac5c84ca2,Nevins St & Schermerhorn St,40.68737247650719,-73.98176059126854,2026-10-18T02:42:08.995Z,1,2.659674352854638,nearest_archived_request_coordinate,static_retrospective_backfill,2026-10-18T02:42:08.995Z,2025-05-01T00:00:00.000Z,2025-05-01T01:00:00.000Z,2025-04-30T12:00:00.000Z,2025-04-30T19:00:00.000Z,2025-05-01T01:00:00.000Z,66.5,0.0,0.0,0.0,6.7,12,1,195,40.675548647131166,-74.00917842775038,40.667835,-73.93768,12,13,false,19,normal_archived_run,data\bronze\weather_forecasts\open_meteo_single_runs\ecmwf_ifs\request_classes=28\forecast_hours=19\year=2025\month=04\ecmwf_ifs_20250430T1200Z.json,2025,5,false,bf65a4cc1d0135537b005d61c076af3695355447c3c13ebd0bec6955b7b077a9,ecmwf_ifs,7,20261018T033852681910Z
2206782596968628766,78 Ave & Queens Blvd S,40.7149,-73.83247,2026-10-18T02:42:08.995Z,1,5.311079210449747,nearest_archived_request_coordinate,static_retrospective_backfill,2026-10-18T02:42:08.995Z,2025-05-01T00:00:00.000

In [0]:
if BUILD_BASE_WEATHER:
    # Reconciling persisted component counts and testing one complete origin slice.
    saved_weather = spark.table(CLASS_TABLE)
    saved_mapping = spark.table(MAPPING_TABLE)
    assert saved_weather.count() == summary.rows
    assert saved_mapping.count() == station_count
    assert_unique(saved_mapping, ["station_id"], "saved station mappings")
    weights = saved_mapping.groupBy("weather_request_class").agg(F.count("*").alias("stations_in_class"))
    weighted_rows = saved_weather.join(weights, "weather_request_class").agg(F.sum("stations_in_class")).first()[0]
    expected_rows = summary.origins * len(HORIZONS) * station_count
    assert weighted_rows == expected_rows
    sample = spark.table(FORECAST_VIEW).filter(F.col("forecast_origin_utc") == F.lit(summary.first_origin))
    assert sample.count() == station_count * len(HORIZONS)
    assert_unique(sample, ["station_id", "forecast_origin_utc", "forecast_horizon"], "sample station forecast keys")
    print("Saved class rows:", summary.rows)
    print("Saved station mappings:", station_count)
    print("Complete station-level row count implied by the validated join:", expected_rows)
    print("Sample origin join passed:", summary.first_origin)


Saved class rows: 2780568
Saved station mappings: 2519
Complete station-level row count implied by the validated join: 250151814
Sample origin join passed: 2024-08-10 07:00:00


In [ ]:
if BUILD_SUBWAY:
    # Reading the completed transfer, without downloading or extracting it again
    subway_package_dir = Path(
        "/Volumes/workspace/citibike_subway/transfer/"
        "subway_disruptions_20261022T215726144524Z"
    )
    subway_transfer_manifest = json.loads(
        (subway_package_dir / "transfer_manifest.json").read_text(encoding="utf-8")
    )
    expected_zip_sha256 = (
        "324472843052ad39265acd9c4281c61d3"
        "db4c30a8b32b705f85f312f3b66143b"
    )
    subway_source_frames = {
        Path(record["path"]).stem: spark.read.parquet(
            str(subway_package_dir / record["path"])
        )
        for record in subway_transfer_manifest["files"]
        if record["path"].startswith("tables/")
        and record["path"].endswith(".parquet")
    }
    bike_subway_bridge = spark.table(BRIDGE_TABLE)


In [0]:
if BUILD_SUBWAY:
    # Selecting unique complex-to-dock links
    dock_links = bike_subway_bridge.select(
        "station_complex_id", "station_id"
    )

    assert dock_links.count() == dock_links.dropDuplicates().count(), (
        "Duplicate complex–dock pairs would multiply exposure rows."
    )

    # Combining attributed exposure with separately retained background loss
    exposure_flags = [
        "lost", "partial_lost", "access_lost", "diverted", "background_loss"
    ]

    attributed_exposure = subway_source_frames["attributed_bins"].select(
        "station_complex_id",
        "bin_utc",
        "lost",
        "partial_lost",
        "access_lost",
        "diverted",
        F.lit(False).alias("background_loss"),
    )

    background_exposure = subway_source_frames["background_bins"].select(
        "station_complex_id",
        "bin_utc",
        F.lit(False).alias("lost"),
        F.lit(False).alias("partial_lost"),
        F.lit(False).alias("access_lost"),
        F.lit(False).alias("diverted"),
        F.lit(True).alias("background_loss"),
    )

    # Restricting exposure to the supported study interval
    study_start_utc = pd.Timestamp(
        subway_transfer_manifest["study_start"]
    ).tz_convert("UTC").to_pydatetime()

    study_end_utc = pd.Timestamp(
        subway_transfer_manifest["study_end_exclusive"]
    ).tz_convert("UTC").to_pydatetime()

    complex_exposure = (
        attributed_exposure.unionByName(background_exposure)
        .filter(
            (F.col("bin_utc") >= F.lit(study_start_utc))
            & (F.col("bin_utc") < F.lit(study_end_utc))
        )
    )

    # Counting each dock-bin once per measure across routes and complexes
    dock_exposure_bins = (
        complex_exposure.join(dock_links, "station_complex_id", "inner")
        .groupBy("station_id", "bin_utc")
        .agg(*[
            F.max(F.col(flag).cast("int")).alias(flag)
            for flag in exposure_flags
        ])
    )

    # Converting flagged bins into retrospective hourly exposure minutes
    dock_hour_exposure = (
        dock_exposure_bins
        .withColumn("hour_utc", F.date_trunc("hour", "bin_utc"))
        .groupBy("station_id", "hour_utc")
        .agg(*[
            (F.sum(flag) * 15).alias(f"{flag}_minutes")
            for flag in exposure_flags
        ])
    )

    # Checking that overlapping links never produce more than 60 minutes
    minute_columns = [f"{flag}_minutes" for flag in exposure_flags]

    invalid_minutes = dock_hour_exposure.filter(
        F.greatest(*[F.col(c) for c in minute_columns]) > 60
    )
    assert invalid_minutes.limit(1).count() == 0, (
        "Hourly exposure exceeds 60 minutes."
    )

    print("Dock-hours with recorded exposure:", dock_hour_exposure.count())
    print(
        "Docks with recorded exposure:",
        dock_hour_exposure.select("station_id").distinct().count(),
    )

    display(
        dock_hour_exposure
        .orderBy("hour_utc", "station_id")
        .limit(10)
    )


Dock-hours with recorded exposure: 299338
Docks with recorded exposure: 324


station_id,hour_utc,lost_minutes,partial_lost_minutes,access_lost_minutes,diverted_minutes,background_loss_minutes
2ad41c1f-5636-4a63-ad52-761e2613061a,2024-07-01T04:00:00.000Z,0,0,0,0,60
5b4a4b20-539b-47c1-9228-1f1f124b3ec9,2024-07-01T04:00:00.000Z,0,0,0,0,60
66db88f5-0aca-11e7-82f6-3863bb44ef7c,2024-07-01T04:00:00.000Z,0,0,0,0,60
66db90f1-0aca-11e7-82f6-3863bb44ef7c,2024-07-01T04:00:00.000Z,0,0,0,0,60
66ddb7bf-0aca-11e7-82f6-3863bb44ef7c,2024-07-01T04:00:00.000Z,0,0,0,0,60
66de0d91-0aca-11e7-82f6-3863bb44ef7c,2024-07-01T04:00:00.000Z,0,0,0,0,60
66de4f9c-0aca-11e7-82f6-3863bb44ef7c,2024-07-01T04:00:00.000Z,0,0,0,0,60
66de88d7-0aca-11e7-82f6-3863bb44ef7c,2024-07-01T04:00:00.000Z,0,0,0,0,60
6fefe1b3-5f0d-47aa-b1dd-cec211239950,2024-07-01T04:00:00.000Z,0,0,0,0,60
aa120a3a-9f2b-4dfc-a88b-384bbb66838a,2024-07-01T04:00:00.000Z,0,0,0,0,60


In [0]:
if BUILD_SUBWAY:
    # Creating one row per real UTC hour in the supported study interval
    subway_coverage_hours = spark.range(1).select(
        F.explode(F.sequence(
            F.lit(study_start_utc).cast("timestamp"),
            F.lit(study_end_utc).cast("timestamp") - F.expr("INTERVAL 1 HOUR"),
            F.expr("INTERVAL 1 HOUR"),
        )).alias("hour_utc")
    ).withColumn(
        "day_ny", F.to_date(F.from_utc_timestamp("hour_utc", "America/New_York"))
    )

    # Retaining the original local-day alert coverage indicators
    daily_alert_coverage = subway_source_frames["alert_coverage_daily"].select(
        F.to_date(
            F.from_utc_timestamp("day_ny", "America/New_York")
        ).alias("day_ny"),
        "alerts_covered",
        "partial_gap",
    )

    # Retaining daily train-count availability
    daily_train_coverage = subway_source_frames["count_log"].select(
        F.to_date("day_ny").alias("day_ny"),
        F.col("status").alias("train_count_status"),
    )

    for label, frame in [
        ("alert coverage", daily_alert_coverage),
        ("train coverage", daily_train_coverage),
    ]:
        assert frame.count() == frame.select("day_ny").distinct().count(), (
            f"Duplicate dates in {label}"
        )

    # Flagging hours overlapping exact UTC alert-gap intervals
    gap_hours = (
        subway_coverage_hours.alias("hours")
        .join(
            F.broadcast(subway_source_frames["alert_gap_intervals"]).alias("gaps"),
            (F.col("hours.hour_utc") < F.col("gaps.end"))
            & (
                F.col("hours.hour_utc") + F.expr("INTERVAL 1 HOUR")
                > F.col("gaps.start")
            ),
            "inner",
        )
        .select(F.col("hours.hour_utc").alias("hour_utc"))
        .distinct()
        .withColumn("utc_alert_gap", F.lit(True))
    )

    # Combining coverage evidence without interpreting unknown hours as normal
    subway_coverage_hours = (
        subway_coverage_hours
        .join(daily_alert_coverage, "day_ny", "left")
        .join(daily_train_coverage, "day_ny", "left")
        .join(gap_hours, "hour_utc", "left")
        .withColumn(
            "alert_coverage_missing",
            F.col("alerts_covered").isNull() | F.col("partial_gap").isNull(),
        )
        .withColumn(
            "alert_gap",
            F.coalesce(F.col("utc_alert_gap"), F.lit(False))
            | F.coalesce(F.col("partial_gap"), F.lit(False))
            | ~F.coalesce(F.col("alerts_covered"), F.lit(False)),
        )
        .withColumn(
            "train_data_missing",
            F.col("train_count_status").isNull()
            | (F.col("train_count_status") == "missing"),
        )
        .drop("utc_alert_gap")
    )

    # Summarizing shared coverage before adding dock-specific uncertainty
    display(
        subway_coverage_hours.agg(
            F.count("*").alias("utc_hours"),
            F.min("hour_utc").alias("first_hour_utc"),
            F.max("hour_utc").alias("last_hour_utc"),
            F.sum(F.col("alert_gap").cast("int")).alias("alert_gap_hours"),
            F.sum(F.col("alert_coverage_missing").cast("int")).alias(
                "hours_without_daily_alert_coverage"
            ),
            F.sum(F.col("train_data_missing").cast("int")).alias(
                "train_data_missing_hours"
            ),
        )
    )


utc_hours,first_hour_utc,last_hour_utc,alert_gap_hours,hours_without_daily_alert_coverage,train_data_missing_hours
18504,2024-07-01T04:00:00.000Z,2026-08-11T03:00:00.000Z,962,0,0


In [ ]:
if BUILD_SUBWAY:
    # Linking the pinned subway routes to mapped docks
    subway_station_routes = (
        subway_source_frames["stations"]
        .select(
            F.col("station_id").alias("subway_stop_id"),
            "station_complex_id",
            F.col("station_name").alias("subway_station_name"),
            F.explode("daytime_routes").alias("route_id"),
        )
    )

    # Reusing the feed assignments recorded in the transfer manifest
    feed_mapping_rows = list(
        subway_transfer_manifest["route_settings"]["route_feed_group"].items()
    )

    route_feed_mapping = spark.createDataFrame(
        feed_mapping_rows,
        "route_id string, feed_group string",
    )

    dock_route_feed_candidates = (
        subway_station_routes
        .join(dock_links, "station_complex_id", "inner")
        .join(route_feed_mapping, "route_id", "left")
        .dropDuplicates()
    )


In [0]:
if BUILD_SUBWAY:
    # Resolving the shared S label using the pinned subway stop IDs
    shuttle_routes_by_stop = {
        "901": "GS",
        "902": "GS",
        "D26": "FS",
        "S01": "FS",
        "S03": "FS",
        "S04": "FS",
    }

    shuttle_route_lookup = F.create_map(*[
        F.lit(value)
        for pair in shuttle_routes_by_stop.items()
        for value in pair
    ])

    # Looking up feed groups after resolving each shuttle service
    dock_route_feed_resolved = (
        dock_route_feed_candidates
        .drop("feed_group")
        .withColumn(
            "resolved_route_id",
            F.when(
                F.col("route_id") == "S",
                shuttle_route_lookup[F.col("subway_stop_id")],
            ).otherwise(F.col("route_id")),
        )
        .join(
            route_feed_mapping.withColumnRenamed(
                "route_id", "resolved_route_id"
            ),
            "resolved_route_id",
            "left",
        )
    )

    assert (
        dock_route_feed_resolved
        .filter(F.col("feed_group").isNull())
        .limit(1).count() == 0
    ), "Some routes still lack a feed-group assignment."

    # Keeping one link per dock and feed group for later quality checks
    dock_feed_bridge = (
        dock_route_feed_resolved
        .select("station_id", "feed_group")
        .distinct()
    )

    assert (
        dock_feed_bridge.select("station_id").distinct().count()
        == dock_links.select("station_id").distinct().count()
    ), "Some mapped docks have no feed-group link."

    # Reviewing the resolved shuttle assignments
    display(
        dock_route_feed_resolved
        .filter(F.col("route_id") == "S")
        .select(
            "subway_stop_id",
            "subway_station_name",
            "resolved_route_id",
            "feed_group",
        )
        .distinct()
        .orderBy("subway_stop_id")
    )

    print("Unresolved route assignments: 0")
    print(
        "Docks linked to feed groups:",
        dock_feed_bridge.select("station_id").distinct().count(),
    )


subway_stop_id,subway_station_name,resolved_route_id,feed_group
901,Grand Central-42 St,GS,1234567S
902,Times Sq-42 St,GS,1234567S
D26,Prospect Park,FS,ACE
S01,Franklin Av,FS,ACE
S03,Park Pl,FS,ACE
S04,Botanic Garden,FS,ACE


Unresolved route assignments: 0
Docks linked to feed groups: 324


In [0]:
if BUILD_SUBWAY:
    # Summarizing the four local-clock feed bins within each hour
    feed_hour_quality = (
        subway_source_frames["feed_health"]
        .withColumn("day_ny", F.to_date("day_ny"))
        .withColumn("local_hour", F.substring("clock", 1, 2).cast("int"))
        .groupBy("day_ny", "local_hour", "feed_group")
        .agg(
            F.countDistinct("clock").alias("feed_bins_present"),
            F.max(
                F.when(F.col("feed_ok") == False, 1).otherwise(0)
            ).alias("feed_problem"),
            F.max(
                F.when(F.col("status") == "unjudged", 1).otherwise(0)
            ).alias("feed_unjudged"),
            F.max(
                F.when(
                    F.col("feed_ok").isNull() | F.col("status").isNull(), 1
                ).otherwise(0)
            ).alias("feed_value_missing"),
        )
    )

    # Identifying repeated local hours without collapsing their UTC identities
    feed_hour_grid = (
        subway_coverage_hours.select("hour_utc", "day_ny")
        .withColumn(
            "local_hour",
            F.hour(F.from_utc_timestamp("hour_utc", "America/New_York")),
        )
        .withColumn(
            "feed_clock_ambiguous",
            F.count("*").over(
                Window.partitionBy("day_ny", "local_hour")
            ) > 1,
        )
    )

    # Checking every feed linked to each dock at each supported hour
    dock_feed_hour_checks = (
        dock_feed_bridge.crossJoin(feed_hour_grid)
        .join(
            feed_hour_quality,
            ["day_ny", "local_hour", "feed_group"],
            "left",
        )
        .withColumn(
            "feed_status_missing",
            F.col("feed_bins_present").isNull()
            | (F.col("feed_bins_present") != 4)
            | (F.col("feed_value_missing") == 1),
        )
    )

    # Retaining the most conservative assessment across a dock's linked feeds
    dock_hour_feed_quality = (
        dock_feed_hour_checks
        .groupBy("station_id", "hour_utc")
        .agg(
            F.max(F.coalesce("feed_problem", F.lit(0)))
            .cast("boolean").alias("feed_problem"),
            F.max(F.coalesce("feed_unjudged", F.lit(0)))
            .cast("boolean").alias("feed_unjudged"),
            F.max(F.col("feed_status_missing").cast("int"))
            .cast("boolean").alias("feed_status_missing"),
            F.max(F.col("feed_clock_ambiguous").cast("int"))
            .cast("boolean").alias("feed_clock_ambiguous"),
        )
    )

    # Reviewing quality flags before combining them with measured exposure
    feed_quality_flags = [
        "feed_problem",
        "feed_unjudged",
        "feed_status_missing",
        "feed_clock_ambiguous",
    ]

    display(
        dock_hour_feed_quality.agg(
            F.count("*").alias("dock_hours"),
            *[
                F.sum(F.col(flag).cast("int")).alias(f"{flag}_dock_hours")
                for flag in feed_quality_flags
            ],
        )
    )


dock_hours,feed_problem_dock_hours,feed_unjudged_dock_hours,feed_status_missing_dock_hours,feed_clock_ambiguous_dock_hours
5995296,118751,53048,0,1296


In [0]:
if BUILD_SUBWAY:
    # Aligning realtime route variants with the station reference
    route_normalization = {
        "6X": "6",
        "7X": "7",
        "FX": "F",
        "SIR": "SI",
        "SS": "SI",
    }

    route_normalization_lookup = F.create_map(*[
        F.lit(value)
        for pair in route_normalization.items()
        for value in pair
    ])

    dock_route_links = (
        dock_route_feed_resolved
        .select(
            "station_id",
            F.coalesce(
                route_normalization_lookup[F.col("resolved_route_id")],
                F.col("resolved_route_id"),
            ).alias("exposure_route_id"),
        )
        .distinct()
    )

    # Retaining scoped episodes that could not be measured
    unmeasured_windows = (
        subway_source_frames["disruption_episodes"]
        .filter(
            F.col("measure_status").isNotNull()
            & (F.col("measure_status") != "measured")
        )
        .select(
            F.explode("disrupted_routes").alias("route_id"),
            F.col("incident_start").alias("start"),
            F.col("alert_end").alias("end"),
            F.lit("unmeasured_episode").alias("reason"),
        )
    )

    # Retaining alert states whose ending update was not observed
    censored_windows = (
        subway_source_frames["active_states"]
        .filter(F.col("end_observed") == False)
        .select(
            F.explode("disrupted_routes").alias("route_id"),
            "start",
            "end",
            F.lit("censored_alert_state").alias("reason"),
        )
    )

    # Clipping retrospective uncertainty windows to the supported study interval
    episode_uncertainty_windows = (
        unmeasured_windows.unionByName(censored_windows)
        .withColumn(
            "exposure_route_id",
            F.coalesce(
                route_normalization_lookup[F.col("route_id")],
                F.col("route_id"),
            ),
        )
        .withColumn("start", F.greatest("start", F.lit(study_start_utc)))
        .withColumn("end", F.least("end", F.lit(study_end_utc)))
        .filter(F.col("start") < F.col("end"))
    )

    # Expanding half-open intervals into overlapping UTC hours
    episode_uncertainty_hours = (
        episode_uncertainty_windows
        .withColumn(
            "hour_utc",
            F.explode(
                F.sequence(
                    F.date_trunc("hour", "start"),
                    F.date_trunc(
                        "hour", F.col("end") - F.expr("INTERVAL 1 MICROSECOND")
                    ),
                    F.expr("INTERVAL 1 HOUR"),
                )
            ),
        )
        .select("exposure_route_id", "hour_utc", "reason")
        .distinct()
    )

    # Combining overlapping route warnings once per dock and hour
    dock_hour_episode_quality = (
        episode_uncertainty_hours
        .join(dock_route_links, "exposure_route_id", "inner")
        .groupBy("station_id", "hour_utc")
        .agg(
            F.max(
                (F.col("reason") == "unmeasured_episode").cast("int")
            ).cast("boolean").alias("unmeasured_episode"),
            F.max(
                (F.col("reason") == "censored_alert_state").cast("int")
            ).cast("boolean").alias("censored_alert_state"),
        )
    )

    display(
        dock_hour_episode_quality.agg(
            F.count("*").alias("dock_hours_with_episode_uncertainty"),
            F.sum(F.col("unmeasured_episode").cast("int")).alias(
                "unmeasured_episode_dock_hours"
            ),
            F.sum(F.col("censored_alert_state").cast("int")).alias(
                "censored_alert_state_dock_hours"
            ),
        )
    )


dock_hours_with_episode_uncertainty,unmeasured_episode_dock_hours,censored_alert_state_dock_hours
121251,62936,69126


In [0]:
if BUILD_SUBWAY:
    # Defining the quality flags used to screen retrospective exposure
    shared_quality_flags = [
        "alert_gap",
        "alert_coverage_missing",
        "train_data_missing",
    ]

    episode_quality_flags = [
        "unmeasured_episode",
        "censored_alert_state",
    ]

    all_quality_flags = (
        shared_quality_flags
        + feed_quality_flags
        + episode_quality_flags
    )

    # Preserving recorded measurements separately from screened values
    recorded_exposure = dock_hour_exposure.select(
        "station_id",
        "hour_utc",
        *[
            F.col(column).alias(f"recorded_{column}")
            for column in minute_columns
        ],
    )

    # Combining shared coverage, dock-specific quality, and recorded exposure
    dock_hour_disruption_panel = (
        dock_hour_feed_quality
        .join(
            subway_coverage_hours.select(
                "hour_utc", "day_ny", *shared_quality_flags
            ),
            "hour_utc",
            "left",
        )
        .join(
            dock_hour_episode_quality,
            ["station_id", "hour_utc"],
            "left",
        )
        .fillna(False, subset=episode_quality_flags)
        .join(
            recorded_exposure,
            ["station_id", "hour_utc"],
            "left",
        )
        .withColumn(
            "has_recorded_exposure",
            F.col("recorded_lost_minutes").isNotNull(),
        )
    )

    # Checking that every panel row has a complete quality assessment
    missing_quality = F.lit(False)
    any_quality_warning = F.lit(False)

    for flag in all_quality_flags:
        missing_quality = missing_quality | F.col(flag).isNull()
        any_quality_warning = any_quality_warning | F.col(flag)

    assert (
        dock_hour_disruption_panel.filter(missing_quality).limit(1).count() == 0
    ), "Some dock-hours lack a quality assessment."

    dock_hour_disruption_panel = dock_hour_disruption_panel.withColumn(
        "exposure_usable",
        ~any_quality_warning,
    )

    # Assigning zero only to screened hours without recorded exposure
    for column in minute_columns:
        dock_hour_disruption_panel = (
            dock_hour_disruption_panel.withColumn(
                column,
                F.when(
                    F.col("exposure_usable"),
                    F.coalesce(F.col(f"recorded_{column}"), F.lit(0)),
                ).otherwise(F.lit(None).cast("long")),
            )
        )

    # Checking that joins preserve the expected dock-hour grain
    expected_dock_hours = (
        dock_links.select("station_id").distinct().count()
        * subway_coverage_hours.count()
    )

    panel_rows = dock_hour_disruption_panel.count()

    assert panel_rows == expected_dock_hours, "Panel row count changed."
    assert (
        dock_hour_disruption_panel
        .select("station_id", "hour_utc").distinct().count()
        == panel_rows
    ), "Duplicate dock-hour keys."

    display(
        dock_hour_disruption_panel.agg(
            F.count("*").alias("dock_hours"),
            F.sum(F.col("exposure_usable").cast("int")).alias("usable_dock_hours"),
            F.round(
                F.avg(F.col("exposure_usable").cast("double")) * 100, 2
            ).alias("usable_percent"),
            F.sum(F.col("has_recorded_exposure").cast("int")).alias(
                "dock_hours_with_recorded_exposure"
            ),
        )
    )


dock_hours,usable_dock_hours,usable_percent,dock_hours_with_recorded_exposure
5995296,5409222,90.22,299338


In [0]:
if BUILD_SUBWAY:
    # Attaching provenance without adding changing timestamps on reruns
    subway_panel_to_save = (
        dock_hour_disruption_panel
        .withColumn(
            "source_transfer_build_id",
            F.lit(subway_transfer_manifest["transfer_build_id"]),
        )
        .withColumn(
            "source_package_sha256",
            F.lit(expected_zip_sha256),
        )
        .withColumn(
            "source_run",
            F.lit(subway_transfer_manifest["source_run"]),
        )
        .withColumn(
            "exposure_interpretation",
            F.lit("retrospective"),
        )
        .withColumn(
            "geography_policy",
            F.lit("nearest_dock_per_subway_location_within_200m"),
        )
    )

    subway_panel_table = (
        "workspace.citibike_subway."
        "archive_v1_station_hour_disruption_retrospective"
    )

    # Preserving an existing table unless its contents match exactly
    if spark.catalog.tableExists(subway_panel_table):
        existing_panel = spark.table(subway_panel_table)

        expected_schema = [
            (field.name, field.dataType.simpleString())
            for field in subway_panel_to_save.schema.fields
        ]
        existing_schema = [
            (field.name, field.dataType.simpleString())
            for field in existing_panel.schema.fields
        ]
        assert existing_schema == expected_schema, (
            "Existing table schema differs; no replacement was performed."
        )

        assert (
            existing_panel.exceptAll(subway_panel_to_save).limit(1).count() == 0
            and subway_panel_to_save.exceptAll(existing_panel).limit(1).count() == 0
        ), "Existing table contents differ; no replacement was performed."

        print("Existing table matches exactly; retained.")

    else:
        # Creating Delta through SQL for compatibility with Spark Connect
        subway_panel_to_save.createOrReplaceTempView(
            "subway_retrospective_panel_to_save"
        )

        spark.sql(f"""
            CREATE TABLE {subway_panel_table}
            USING DELTA
            COMMENT 'Retrospective dock-hour subway exposure with quality flags.
                     Target-hour exposure is not a forecast-origin predictor.'
            AS SELECT * FROM subway_retrospective_panel_to_save
        """)

        print("Created:", subway_panel_table)

    # Verifying the persisted row count and quality-screen result
    saved_subway_summary = (
        spark.table(subway_panel_table)
        .agg(
            F.count("*").alias("dock_hours"),
            F.sum(F.col("exposure_usable").cast("long")).alias(
                "usable_dock_hours"
            ),
        )
        .first()
    )

    assert saved_subway_summary["dock_hours"] == 5_995_296
    assert saved_subway_summary["usable_dock_hours"] == 5_409_222

    print("Saved dock-hours:", saved_subway_summary["dock_hours"])
    print("Saved usable dock-hours:", saved_subway_summary["usable_dock_hours"])


Created: workspace.citibike_subway.archive_v1_station_hour_disruption_retrospective
Saved dock-hours: 5995296
Saved usable dock-hours: 5409222


In [0]:
if BUILD_SUBWAY:
    # Assigning publications to subsequent hourly origins within three hours
    alert_origin_candidates = (
        subway_source_frames["alerts"]
        .select(
            "alert_id", "event_id", "update_number",
            F.col("date").alias("publication_time_utc"),
            "status_label", "affected",
        )
        .withColumn(
            "forecast_origin_utc",
            F.explode(
                F.sequence(
                    F.date_trunc("hour", "publication_time_utc")
                    + F.expr("INTERVAL 1 HOUR"),
                    F.date_trunc("hour", "publication_time_utc")
                    + F.expr("INTERVAL 3 HOURS"),
                    F.expr("INTERVAL 1 HOUR"),
                )
            ),
        )
        .filter(
            (F.col("forecast_origin_utc") >= F.lit(study_start_utc))
            & (F.col("forecast_origin_utc") < F.lit(study_end_utc))
            & (
                F.col("forecast_origin_utc")
                < F.col("publication_time_utc") + F.expr("INTERVAL 3 HOURS")
            )
        )
    )

    # Keeping the latest available publication before interpreting its status
    latest_event_order = Window.partitionBy(
        "event_id", "forecast_origin_utc"
    ).orderBy(
        F.col("publication_time_utc").desc(),
        F.col("update_number").desc(),
        F.col("alert_id").desc(),
    )

    recent_event_states = (
        alert_origin_candidates
        .withColumn("publication_rank", F.row_number().over(latest_event_order))
        .filter(F.col("publication_rank") == 1)
        .drop("publication_rank")
        .withColumn(
            "publication_age_minutes",
            (
                F.unix_timestamp("forecast_origin_utc")
                - F.unix_timestamp("publication_time_utc")
            ) / 60.0,
        )
    )

    print("Recent event–origin rows:", recent_event_states.count())


Recent event–origin rows: 145848


In [0]:
if BUILD_SUBWAY:
    # Defining numeric indicators from structured MTA status labels
    alert_status_flags = {
        "status_reroute": "reroute",
        "status_suspended": "suspended",
        "status_part_suspended": "part-suspended",
        "status_stops_skipped": "stops-skipped",
        "status_severe_delays": "severe-delays",
        "status_delays": "delays",
        "status_cancellations": "cancellations",
        "status_express_to_local": "express-to-local",
        "status_boarding_change": "boarding-change",
        "status_information_outage": "information-outage",
    }

    # Splitting combined labels while retaining every selected event state
    recent_event_features = recent_event_states.withColumn(
        "status_components",
        F.transform(
            F.split(F.lower(F.coalesce("status_label", F.lit(""))), r"\|"),
            lambda component: F.trim(component),
        ),
    )

    for column, status in alert_status_flags.items():
        recent_event_features = recent_event_features.withColumn(
            column,
            F.array_contains("status_components", status).cast("int"),
        )

    # Summarizing the indicators across event–origin rows
    display(
        recent_event_features.agg(*[
            F.sum(column).alias(column)
            for column in alert_status_flags
        ])
    )


status_reroute,status_suspended,status_part_suspended,status_stops_skipped,status_severe_delays,status_delays,status_cancellations,status_express_to_local,status_boarding_change,status_information_outage
1594,228,1307,3087,505,139154,326,1872,240,208


In [0]:
if BUILD_SUBWAY:
    # Reading route symbols from each origin's selected publication
    alert_route_candidates = (
        recent_event_features
        .withColumn(
            "alert_route_symbol",
            F.explode(F.split(F.coalesce("affected", F.lit("")), r"\|")),
        )
        .withColumn("alert_route_symbol", F.trim("alert_route_symbol"))
        .filter(F.col("alert_route_symbol") != "")
        .withColumn(
            "route_mapping_ambiguous",
            (F.col("alert_route_symbol") == "S").cast("int"),
        )
        .withColumn(
            "exposure_route_id",
            F.explode(
                F.when(
                    F.col("alert_route_symbol") == "S",
                    F.array(F.lit("GS"), F.lit("FS"), F.lit("H")),
                ).otherwise(
                    F.array(
                        F.coalesce(
                            route_normalization_lookup[
                                F.col("alert_route_symbol")
                            ],
                            F.col("alert_route_symbol"),
                        )
                    )
                )
            ),
        )
    )

    # Linking routes to docks and counting each event once per dock and origin
    dock_event_features = (
        alert_route_candidates
        .join(dock_route_links, "exposure_route_id", "inner")
        .groupBy("station_id", "forecast_origin_utc", "event_id")
        .agg(
            F.max("publication_time_utc").alias("publication_time_utc"),
            F.min("publication_age_minutes").alias("publication_age_minutes"),
            F.max("route_mapping_ambiguous").alias("route_mapping_ambiguous"),
            *[
                F.max(column).alias(column)
                for column in alert_status_flags
            ],
        )
    )

    display(
        dock_event_features.agg(
            F.count("*").alias("dock_event_origin_rows"),
            F.countDistinct("station_id").alias("linked_docks"),
            F.sum("route_mapping_ambiguous").alias(
                "rows_with_ambiguous_shuttle_links"
            ),
        )
    )


dock_event_origin_rows,linked_docks,rows_with_ambiguous_shuttle_links
6068912,324,0


In [0]:
if BUILD_SUBWAY:
    # Naming counts explicitly as recently published event statuses
    origin_status_count_columns = {
        column: f"recent_{column.removeprefix('status_')}_event_count"
        for column in alert_status_flags
    }

    # Aggregating distinct linked events into dock-level forecasting features
    dock_origin_alert_features_sparse = (
        dock_event_features
        .groupBy("station_id", "forecast_origin_utc")
        .agg(
            F.count("*").alias("recent_event_count"),
            F.min("publication_age_minutes").alias(
                "latest_publication_age_minutes"
            ),
            F.max("publication_time_utc").alias(
                "latest_publication_time_utc"
            ),
            F.max("route_mapping_ambiguous").alias(
                "has_ambiguous_shuttle_link"
            ),
            *[
                F.sum(source_column).alias(output_column)
                for source_column, output_column
                in origin_status_count_columns.items()
            ],
        )
    )

    display(
        dock_origin_alert_features_sparse.agg(
            F.count("*").alias("dock_origins_with_recent_publications"),
            F.max("recent_event_count").alias("maximum_events_per_dock_origin"),
            F.min("latest_publication_age_minutes").alias(
                "minimum_publication_age_minutes"
            ),
            F.max("latest_publication_age_minutes").alias(
                "maximum_latest_publication_age_minutes"
            ),
        )
    )


dock_origins_with_recent_publications,maximum_events_per_dock_origin,minimum_publication_age_minutes,maximum_latest_publication_age_minutes
3097558,22,1.0,179.0


In [0]:
if BUILD_SUBWAY:
    # Measuring archive activity using only publications before each origin
    origin_archive_activity = (
        alert_origin_candidates
        .groupBy("forecast_origin_utc")
        .agg(
            F.count("*").alias("archive_publications_last_3h"),
        )
    )

    # Reusing only the UTC clock grid, without retrospective quality flags
    forecast_origin_grid = subway_coverage_hours.select(
        F.col("hour_utc").alias("forecast_origin_utc")
    )

    dock_origin_alert_features = (
        dock_links.select("station_id").distinct()
        .crossJoin(forecast_origin_grid)
        .join(origin_archive_activity, "forecast_origin_utc", "left")
        .join(
            dock_origin_alert_features_sparse,
            ["station_id", "forecast_origin_utc"],
            "left",
        )
        .fillna(0, subset=["archive_publications_last_3h"])
        .withColumn(
            "archive_silence_last_3h",
            F.col("archive_publications_last_3h") == 0,
        )
        .withColumn(
            "has_recent_route_publication",
            F.col("recent_event_count").isNotNull(),
        )
    )

    # Filling absent route counts only when recent archive activity exists
    origin_count_columns = [
        "recent_event_count",
        *origin_status_count_columns.values(),
    ]

    for column in origin_count_columns:
        dock_origin_alert_features = dock_origin_alert_features.withColumn(
            column,
            F.when(
                ~F.col("archive_silence_last_3h"),
                F.coalesce(F.col(column), F.lit(0)),
            ).otherwise(F.lit(None).cast("long")),
        )

    # Retaining ambiguity as a property of the publications actually linked
    dock_origin_alert_features = dock_origin_alert_features.fillna(
        0, subset=["has_ambiguous_shuttle_link"]
    )

    display(
        dock_origin_alert_features.agg(
            F.count("*").alias("dock_origins"),
            F.sum(F.col("has_recent_route_publication").cast("long")).alias(
                "with_recent_route_publications"
            ),
            F.sum(F.col("archive_silence_last_3h").cast("long")).alias(
                "with_archive_silence"
            ),
        )
    )


dock_origins,with_recent_route_publications,with_archive_silence
5995296,3097558,278316


In [0]:
if BUILD_SUBWAY:
    # Checking unique keys and publication timing before saving
    alert_feature_checks = dock_origin_alert_features.agg(
        F.count("*").alias("rows"),
        F.countDistinct(
            F.struct("station_id", "forecast_origin_utc")
        ).alias("unique_keys"),
        F.sum(
            F.when(
                F.col("station_id").isNull()
                | F.col("forecast_origin_utc").isNull(),
                1,
            ).otherwise(0)
        ).alias("missing_keys"),
        F.sum(
            F.when(
                F.col("latest_publication_time_utc")
                >= F.col("forecast_origin_utc"),
                1,
            ).otherwise(0)
        ).alias("invalid_publication_times"),
    ).first()

    assert alert_feature_checks["rows"] == 5_995_296
    assert alert_feature_checks["unique_keys"] == alert_feature_checks["rows"]
    assert alert_feature_checks["missing_keys"] == 0
    assert alert_feature_checks["invalid_publication_times"] == 0

    # Recording the assumptions needed to reproduce these features
    alert_features_to_save = (
        dock_origin_alert_features
        .withColumn(
            "source_transfer_build_id",
            F.lit(subway_transfer_manifest["transfer_build_id"]),
        )
        .withColumn("source_package_sha256", F.lit(expected_zip_sha256))
        .withColumn("publication_window_hours", F.lit(3))
        .withColumn("assumed_ingestion_delay_minutes", F.lit(0))
        .withColumn(
            "geography_policy",
            F.lit("retrospective_nearest_dock_within_200m"),
        )
    )

    alert_feature_table = (
        "workspace.citibike_subway."
        "archive_v1_station_origin_alert_features"
    )

    # Creating a separate table without overwriting an existing version
    assert not spark.catalog.tableExists(alert_feature_table), (
        "The destination already exists. No replacement was performed."
    )

    alert_features_to_save.createOrReplaceTempView(
        "subway_origin_alert_features_to_save"
    )

    spark.sql(f"""
        CREATE TABLE {alert_feature_table}
        USING DELTA
        COMMENT 'Recent route-associated MTA publications available strictly
                 before each origin. Three-hour window and zero ingestion
                 delay assumed. Uses retrospective station geography.'
        AS SELECT * FROM subway_origin_alert_features_to_save
    """)

    # Confirming that all validated rows were persisted
    saved_alert_rows = spark.table(alert_feature_table).count()
    assert saved_alert_rows == alert_feature_checks["rows"]

    print("Created:", alert_feature_table)
    print("Saved dock-origins:", saved_alert_rows)


Created: workspace.citibike_subway.archive_v1_station_origin_alert_features
Saved dock-origins: 5995296


In [0]:
if BUILD_WEATHER_EXTENSION:
    # Locating the uploaded weather extension
    weather_extension_zip = Path(
        "/Volumes/workspace/citibike_weather/transfer/"
        "citibike_weather_extension_20261023T165211825083Z.zip"
    )

    expected_extension_sha256 = (
        "a27a1c5d63d3ead902894fd669a7457851"
        "d17c84bfc287bf6da67c9b69fa1051"
    )

    assert weather_extension_zip.is_file(), (
        f"Uploaded ZIP not found: {weather_extension_zip}"
    )

    # Checking that the uploaded package matches the local package
    with weather_extension_zip.open("rb") as handle:
        actual_extension_sha256 = hashlib.file_digest(
            handle, "sha256"
        ).hexdigest()

    assert actual_extension_sha256 == expected_extension_sha256, (
        "Uploaded ZIP hash differs from the local package."
    )

    weather_extension_root = (
        weather_extension_zip.parent / "inputs" / actual_extension_sha256
    )

    # Verifying all package members before extracting any files
    with zipfile.ZipFile(weather_extension_zip) as archive:
        member_names = archive.namelist()
        assert len(member_names) == len(set(member_names))

        weather_extension_manifest = json.loads(
            archive.read("input_manifest.json")
        )
        extension_records = weather_extension_manifest["files"]

        assert weather_extension_manifest["format_version"] == 1
        assert weather_extension_manifest["package_type"] == (
            "weather_partition_extension"
        )
        assert len(extension_records) == 4
        assert sum(record["rows"] for record in extension_records) == 371_616
        assert set(member_names) == {
            record["path"] for record in extension_records
        } | {"input_manifest.json"}

        verified_payloads = {}

        for record in extension_records:
            relative = Path(record["path"])
            assert not relative.is_absolute()
            assert ".." not in relative.parts
            assert relative.parts[0] == "weather_classes"
            assert relative.suffix == ".parquet"

            payload = archive.read(record["path"])
            assert len(payload) == record["size"]
            assert hashlib.sha256(payload).hexdigest() == record["sha256"]

            verified_payloads[relative] = payload

    # Preserving existing extracted files when rerunning this cell
    for relative, payload in verified_payloads.items():
        destination = weather_extension_root / relative
        destination.parent.mkdir(parents=True, exist_ok=True)

        if destination.exists():
            assert destination.read_bytes() == payload, (
                f"Existing extracted file differs: {destination}"
            )
        else:
            with destination.open("xb") as handle:
                handle.write(payload)

    print("Uploaded ZIP hash matched.")
    print("Verified and extracted monthly files:", len(extension_records))
    print("Packaged rows:", sum(record["rows"] for record in extension_records))
    print("Extension input:", weather_extension_root / "weather_classes")
    print("No Delta tables changed.")


Uploaded ZIP hash matched.
Verified and extracted monthly files: 4
Packaged rows: 371616
Extension input: /Volumes/workspace/citibike_weather/transfer/inputs/a27a1c5d63d3ead902894fd669a7457851d17c84bfc287bf6da67c9b69fa1051/weather_classes
No Delta tables changed.


In [0]:
if BUILD_WEATHER_EXTENSION:
    spark.conf.set("spark.sql.session.timeZone", "UTC")

    # Reading the existing class table and the four replacement partitions
    base_weather = spark.table(
        "workspace.citibike_weather.archive_v1_weather_classes"
    )

    incoming_weather = (
        spark.read
        .option("recursiveFileLookup", "true")
        .parquet(str(weather_extension_root / "weather_classes"))
    )

    # Matching the original data columns and Spark types
    build_metadata_columns = {
        "input_package_sha256",
        "weather_model",
        "assumed_availability_delay_hours",
        "dataset_build_id",
    }

    weather_data_fields = [
        field for field in base_weather.schema.fields
        if field.name not in build_metadata_columns
    ]
    weather_data_columns = [field.name for field in weather_data_fields]

    assert set(weather_data_columns) <= set(incoming_weather.columns)

    incoming_weather = incoming_weather.select(*[
        F.col(field.name).cast(field.dataType).alias(field.name)
        for field in weather_data_fields
    ])

    assert incoming_weather.count() == 371_616

    # Confirming that the replacement preserves all previously saved June data
    replacement_months = incoming_weather.select(
        "origin_year", "origin_month"
    ).distinct()

    old_replaced_weather = (
        base_weather
        .join(replacement_months, ["origin_year", "origin_month"], "inner")
        .select(*weather_data_columns)
    )

    assert (
        old_replaced_weather.exceptAll(incoming_weather).limit(1).count() == 0
    ), "Some existing weather rows differ or are missing from the replacement."

    # Retaining unaffected months and recording the extension's package provenance
    retained_weather = base_weather.join(
        replacement_months,
        ["origin_year", "origin_month"],
        "left_anti",
    )

    incoming_weather = (
        incoming_weather
        .withColumn("input_package_sha256", F.lit(actual_extension_sha256))
        .withColumn("weather_model", F.lit("ecmwf_ifs"))
        .withColumn("assumed_availability_delay_hours", F.lit(7))
        .withColumn(
            "dataset_build_id",
            F.lit(weather_extension_manifest["extension_build_id"]),
        )
    )

    combined_weather = retained_weather.unionByName(incoming_weather)

    # Checking the combined dataset's size, unique keys, and hourly coverage
    combined_summary = combined_weather.agg(
        F.count("*").alias("rows"),
        F.countDistinct(F.struct(
            "forecast_origin_utc",
            "forecast_horizon",
            "weather_request_class",
        )).alias("unique_keys"),
        F.countDistinct("forecast_origin_utc").alias("origins"),
        F.min("forecast_origin_utc").alias("first_origin"),
        F.max("forecast_origin_utc").alias("last_origin"),
    ).first()

    assert combined_summary["rows"] == 3_031_560
    assert combined_summary["unique_keys"] == combined_summary["rows"]
    assert combined_summary["origins"] == 18_045
    assert (
        int((
            combined_summary["last_origin"]
            - combined_summary["first_origin"]
        ).total_seconds() / 3600) + 1
        == combined_summary["origins"]
    )

    print("Existing June data preserved.")
    print("Combined class rows:", combined_summary["rows"])
    print("Hourly origins:", combined_summary["origins"])
    print("First origin UTC:", combined_summary["first_origin"])
    print("Last origin UTC:", combined_summary["last_origin"])
    print("Combined checks passed. No Delta tables changed.")


Existing June data preserved.
Combined class rows: 3031560
Hourly origins: 18045
First origin UTC: 2024-08-10 07:00:00
Last origin UTC: 2026-09-01 03:00:00
Combined checks passed. No Delta tables changed.


In [0]:
if BUILD_WEATHER_EXTENSION:
    # Defining the new weather version without replacing archive_v1
    v2_class_table = "workspace.citibike_weather.archive_v2_weather_classes"
    v2_mapping_table = (
        "workspace.citibike_weather.archive_v2_station_weather_mapping"
    )
    v2_forecast_view = "workspace.citibike_weather.archive_v2_station_forecast"

    v2_build_id = weather_extension_manifest["extension_build_id"]

    # Checking all destination names before creating any objects
    for destination in (
        v2_class_table, v2_mapping_table, v2_forecast_view
    ):
        assert not spark.catalog.tableExists(destination), (
            f"Destination already exists; no replacement performed: {destination}"
        )

    # Preserving source build identifiers while assigning one combined build ID
    v2_weather = (
        combined_weather
        .withColumn("source_dataset_build_id", F.col("dataset_build_id"))
        .withColumn("dataset_build_id", F.lit(v2_build_id))
    )

    v2_mapping = (
        spark.table(
            "workspace.citibike_weather.archive_v1_station_weather_mapping"
        )
        .withColumn("source_dataset_build_id", F.col("dataset_build_id"))
        .withColumn("dataset_build_id", F.lit(v2_build_id))
    )

    v2_weather.createOrReplaceTempView("citibike_v2_weather_to_save")
    v2_mapping.createOrReplaceTempView("citibike_v2_mapping_to_save")

    # Saving the compact weather and mapping components
    spark.sql(f"""
        CREATE TABLE {v2_class_table}
        USING DELTA
        PARTITIONED BY (origin_year, origin_month)
        AS SELECT * FROM citibike_v2_weather_to_save
    """)

    spark.sql(f"""
        CREATE TABLE {v2_mapping_table}
        USING DELTA
        AS SELECT * FROM citibike_v2_mapping_to_save
    """)

    # Reusing the original station-level view structure
    station_mapping_columns = [
        "station_id",
        "station_name",
        "station_latitude",
        "station_longitude",
        "station_metadata_fetched_at_utc",
        "station_location_version_count",
        "weather_mapping_distance_km",
        "weather_mapping_method",
        "station_geography_policy",
        "station_snapshot_cutoff_utc",
    ]

    mapping_projection = ", ".join(
        f"m.`{column}`" for column in station_mapping_columns
    )

    spark.sql(f"""
        CREATE VIEW {v2_forecast_view} AS
        SELECT {mapping_projection}, w.*
        FROM {v2_mapping_table} m
        JOIN {v2_class_table} w
          ON m.weather_request_class = w.weather_request_class
         AND m.dataset_build_id = w.dataset_build_id
    """)

    # Checking saved components and one complete station forecast slice
    saved_v2_class_rows = spark.table(v2_class_table).count()
    saved_v2_station_count = spark.table(v2_mapping_table).count()

    assert saved_v2_class_rows == 3_031_560
    assert saved_v2_station_count == 2_519

    last_origin_station_rows = (
        spark.table(v2_forecast_view)
        .filter(
            F.col("forecast_origin_utc")
            == F.lit(combined_summary["last_origin"])
        )
        .count()
    )
    assert last_origin_station_rows == saved_v2_station_count * 6

    print("Saved weather:", v2_class_table)
    print("Saved mapping:", v2_mapping_table)
    print("Station forecast view:", v2_forecast_view)
    print("Saved class rows:", saved_v2_class_rows)
    print("Mapped stations:", saved_v2_station_count)
    print("Station forecast rows at final origin:", last_origin_station_rows)
    print("archive_v1 retained unchanged.")


Saved weather: workspace.citibike_weather.archive_v2_weather_classes
Saved mapping: workspace.citibike_weather.archive_v2_station_weather_mapping
Station forecast view: workspace.citibike_weather.archive_v2_station_forecast
Saved class rows: 3031560
Mapped stations: 2519
Station forecast rows at final origin: 15114
archive_v1 retained unchanged.


In [ ]:
# Loading the saved datasets for the next Team 6 step
saved_dataset_names = {
    "weather_classes": V2_CLASS_TABLE,
    "weather_mapping": V2_MAPPING_TABLE,
    "weather_forecasts": V2_FORECAST_VIEW,
    "subway_bridge": BRIDGE_TABLE,
    "subway_alert_features": ALERT_FEATURE_TABLE,
    "retrospective_exposure": RETROSPECTIVE_TABLE,
}
missing_saved_tables = [
    name for name in saved_dataset_names.values()
    if not spark.catalog.tableExists(name)
]
assert not missing_saved_tables, (
    f"Saved inputs missing from this workspace: {missing_saved_tables}. "
    "Use the shared workspace or import the compact handoff first."
)
saved_frames = {
    label: spark.table(name) for label, name in saved_dataset_names.items()
}
weather_classes = saved_frames["weather_classes"]
weather_mapping = saved_frames["weather_mapping"]
weather_forecasts = saved_frames["weather_forecasts"]
subway_bridge = saved_frames["subway_bridge"]
subway_alert_features = saved_frames["subway_alert_features"]
retrospective_exposure = saved_frames["retrospective_exposure"]

# Keeping the datasets seperate, without joining demand or filling unmapped docks
for label, name in saved_dataset_names.items():
    print(f"{label}: {name}")
print("Study start UTC:", STUDY_START_UTC)
print("Exclusive endpoint UTC:", STUDY_END_UTC_EXCLUSIVE)
print("Last included New York day: 2026-08-10")
print("Weather horizons stored: 1-6; proposed modelling horizons: 1-3")
print("Keeping station IDs, target definitions and split purging for the modelling step.")


In [ ]:
# Keeping the portable table export optional on normal runs
if EXPORT_HANDOFF:
    from pathlib import Path
    from datetime import datetime, timezone
    import hashlib
    import json
    import zipfile

    # Exporting the five saved tables without expanding station weather
    handoff_build_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    handoff_root = Path(
        "/Volumes/workspace/citibike_weather/transfer/exports"
    ) / f"citibike_weather_subway_handoff_{handoff_build_id}"
    handoff_root.mkdir(parents=True, exist_ok=False)
    spark.conf.set("spark.sql.session.timeZone", "UTC")

    handoff_specs = {
        "weather_classes": (
            "workspace.citibike_weather.archive_v2_weather_classes",
            3_031_560,
            ["weather_request_class", "forecast_origin_utc", "forecast_horizon"],
        ),
        "station_weather_mapping": (
            "workspace.citibike_weather.archive_v2_station_weather_mapping",
            2_519, ["station_id"],
        ),
        "subway_bridge": (
            "workspace.citibike_subway.subway_complex_bike_bridge_200m_v1",
            331, ["station_complex_id", "station_id"],
        ),
        "subway_alert_features": (
            "workspace.citibike_subway.archive_v1_station_origin_alert_features",
            5_995_296, ["station_id", "forecast_origin_utc"],
        ),
        "subway_retrospective": (
            "workspace.citibike_subway.archive_v1_station_hour_disruption_retrospective",
            5_995_296, ["station_id", "hour_utc"],
        ),
    }

    handoff_manifest = {
        "format_version": 1,
        "build_id": handoff_build_id,
        "study_start_utc": "2025-03-01T05:00:00Z",
        "study_end_utc_exclusive": "2026-08-11T04:00:00Z",
        "stored_weather_horizons": [1, 2, 3, 4, 5, 6],
        "proposed_modeling_horizons": [1, 2, 3],
        "coverage_policy": "full_saved_archives; filter to common study period later",
        "datasets": {},
        "files": [],
    }

    # Pinning each Delta snapshot before writing its portable Parquet files
    for label, (table_name, expected_rows, keys) in handoff_specs.items():
        version = int(spark.sql(
            f"DESCRIBE HISTORY {table_name} LIMIT 1"
        ).first()["version"])
        frame = spark.sql(
            f"SELECT * FROM {table_name} VERSION AS OF {version}"
        )
        relative_folder = f"tables/{label}"
        export_path = str(handoff_root / relative_folder)

        (
            frame.repartition(4 if expected_rows > 100_000 else 1)
            .write.mode("error")
            .option("compression", "zstd")
            .parquet(export_path)
        )
        exported_rows = spark.read.parquet(export_path).count()
        assert exported_rows == expected_rows, f"Export row count differs: {label}"

        handoff_manifest["datasets"][label] = {
            "path": relative_folder,
            "source_table": table_name,
            "source_delta_version": version,
            "rows": exported_rows,
            "primary_key": keys,
            "schema": frame.schema.jsonValue(),
        }
        print(f"Exported {label}: {exported_rows:,} rows")

    # Recording hashes before creating the final transfer ZIP
    for path in sorted(handoff_root.rglob("*.parquet")):
        with path.open("rb") as handle:
            file_hash = hashlib.file_digest(handle, "sha256").hexdigest()
        handoff_manifest["files"].append({
            "path": path.relative_to(handoff_root).as_posix(),
            "size_bytes": path.stat().st_size,
            "sha256": file_hash,
        })

    (handoff_root / "transfer_manifest.json").write_text(
        json.dumps(handoff_manifest, indent=2), encoding="utf-8"
    )
    # Building the ZIP locally because Volumes do not support ZIP random writes
    import tempfile
    import shutil

    handoff_zip = handoff_root.with_suffix(".zip")
    with tempfile.TemporaryDirectory(prefix="citibike_handoff_") as temp_folder:
        local_zip = Path(temp_folder) / handoff_zip.name
        with zipfile.ZipFile(local_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
            for record in handoff_manifest["files"]:
                archive.write(handoff_root / record["path"], record["path"])
            archive.write(handoff_root / "transfer_manifest.json", "transfer_manifest.json")
        with local_zip.open("rb") as handle:
            local_zip_hash = hashlib.file_digest(handle, "sha256").hexdigest()
        with local_zip.open("rb") as source_handle:
            with handoff_zip.open("xb") as destination_handle:
                shutil.copyfileobj(source_handle, destination_handle, length=1024 * 1024)

    with handoff_zip.open("rb") as handle:
        handoff_zip_hash = hashlib.file_digest(handle, "sha256").hexdigest()
    assert handoff_zip_hash == local_zip_hash, "ZIP copy hash mismatch."

    print("ZIP:", handoff_zip)
    print("Size (MiB):", round(handoff_zip.stat().st_size / 2**20, 2))
    print("ZIP SHA-256:", handoff_zip_hash)
    print("Finished ZIP copied and verified; original Delta tables retained.")
